In [ ]:
%load_ext autoreload
%autoreload 2

# torch.profiler: Kernel-Level GPU Profiling

Tier 2 deep-investigation profiling for the mermaid-segmentation training loop.
Complements the wall-clock timing already in `train_epoch` (`data_loading_sec`,
`forward_sec`, `backward_sec`) by tracing every CUDA kernel, producing operator-level
timing tables, memory allocation profiles, and exportable Chrome / TensorBoard traces.

**Default:** `PROFILE_DATA_SOURCE=synthetic` — no AWS required.

**SageMaker / S3:** set `PROFILE_DATA_SOURCE=s3` to profile a real `MermaidDataset` `DataLoader`
(S3 GetObject, parquet, Albumentations, `collate_fn`). The execution role needs `s3:GetObject` on the
dataset bucket; if loads fail, run `aws sts get-caller-identity` before debugging IAM.


## 1. Setup

In [1]:
import copy
import os
from datetime import datetime
from pathlib import Path

import albumentations as A
import torch
import torch.nn.functional as F
from torch.utils import benchmark
from torch.utils.data import DataLoader

import mermaidseg.datasets.dataset
from mermaidseg.io import setup_config
from mermaidseg.model.meta import MetaModel

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device:     {device}")

Device:     cpu


## 2. Configuration

In [3]:
cfg = setup_config(
    config_path="../../configs/benchmark.yaml",
    config_base_path="../../configs/base_mermaid.yaml",
)
cfg.training.epochs = 1

# Deep-copy before MetaModel pops keys from the dicts
cfg_model = copy.deepcopy(cfg.model)
cfg_training = copy.deepcopy(cfg.training)

In [4]:
print(f"Model:         {cfg_model.name}")
print(f"Encoder:       {cfg_model.encoder_name}")
print(f"Input size:    {cfg_model.input_size}")
print(f"Training mode: {cfg.training_mode}")
print(f"Epochs:        {cfg_training.epochs}")

Model:         LinearDINOv3
Encoder:       facebook/dinov3-vitb16-pretrain-lvd1689m
Input size:    [512, 512]
Training mode: standard
Epochs:        1


## 2b. Data source and profiler output directory

| Variable | Default | Meaning |
|----------|---------|---------|
| `PROFILE_DATA_SOURCE` | `synthetic` | `synthetic` or `s3` |
| `PROFILE_BATCH_SIZE` | `2` | Batch size |
| `PROFILE_NUM_WORKERS` | `0` | DataLoader workers in S3 mode (`>0` closer to training jobs) |
| `PROFILE_N_BATCHES` | `10` | Batches to prefetch in S3 mode |
| `PROFILER_LOG_DIR` | (see below) | Override trace output directory |

On SageMaker **training** jobs, traces default to `$SM_OUTPUT_DATA_DIR/profiler_logs` when `PROFILER_LOG_DIR` is unset.
Copy artifacts locally or run `aws s3 sync` to your bucket to open TensorBoard on a laptop.


In [5]:
_override = os.environ.get("PROFILER_LOG_DIR")
if _override:
    PROFILER_LOG_DIR = Path(_override)
elif os.environ.get("SM_OUTPUT_DATA_DIR"):
    PROFILER_LOG_DIR = Path(os.environ["SM_OUTPUT_DATA_DIR"]) / "profiler_logs"
else:
    PROFILER_LOG_DIR = Path("./profiler_logs")
PROFILER_LOG_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_DATA_SOURCE = os.environ.get("PROFILE_DATA_SOURCE", "synthetic").strip().lower()
PROFILE_NUM_WORKERS = int(os.environ.get("PROFILE_NUM_WORKERS", "0"))
BATCH_SIZE = int(os.environ.get("PROFILE_BATCH_SIZE", "2"))

print(f"PROFILER_LOG_DIR:      {PROFILER_LOG_DIR.resolve()}")
print(f"PROFILE_DATA_SOURCE:   {PROFILE_DATA_SOURCE!r}")
print(f"BATCH_SIZE:            {BATCH_SIZE}")
if os.environ.get("SM_OUTPUT_DATA_DIR"):
    print(f"SM_OUTPUT_DATA_DIR:    {os.environ['SM_OUTPUT_DATA_DIR']}")

PROFILER_LOG_DIR:      /Users/laurenyee/Projects/mermaid-segmentation/nbs/nb_experiments/profiler_logs
PROFILE_DATA_SOURCE:   'synthetic'
BATCH_SIZE:            2


## 3. Data (synthetic or S3)

Synthetic batches profile GPU compute only. S3 mode uses merged `benchmark.yaml` + `base_mermaid.yaml`
`cfg.data` and train augmentations so `data_loading` includes real I/O and collation.


In [6]:
H, W = cfg_model.input_size
WAIT, WARMUP, ACTIVE = 1, 1, 3
N_WARMUP = 5
N_BATCHES = max(10, int(os.environ.get("PROFILE_N_BATCHES", "10")))
_need_batches = max(N_WARMUP, WAIT + WARMUP + ACTIVE)


def prepare_batch(batch):
    x, y = batch
    if x.device == device:
        return batch
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)


if PROFILE_DATA_SOURCE == "synthetic":
    _cs = getattr(cfg.data, "class_subset", None)
    NUM_CLASSES = len(_cs) + 1 if _cs is not None else 16
    profile_batches = [
        (
            torch.randn(BATCH_SIZE, 3, H, W, device=device),
            torch.randint(0, NUM_CLASSES, (BATCH_SIZE, H, W), dtype=torch.long, device=device),
        )
        for _ in range(N_BATCHES)
    ]
    print(f"Synthetic: {N_BATCHES} batches  batch_size={BATCH_SIZE}  num_classes={NUM_CLASSES}")
elif PROFILE_DATA_SOURCE == "s3":
    data_kw = copy.deepcopy(dict(cfg.data))
    dataset_name = data_kw.pop("name")
    data_kw.pop("batch_size", None)
    transform_train = A.Compose([getattr(A, name)(**params) for name, params in cfg.augmentation["train"].items()])
    train_ds = getattr(mermaidseg.datasets.dataset, dataset_name)(
        transform=transform_train,
        **data_kw,
    )
    NUM_CLASSES = train_ds.num_classes
    s3_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=PROFILE_NUM_WORKERS,
        drop_last=True,
        collate_fn=train_ds.collate_fn,
        pin_memory=torch.cuda.is_available(),
    )
    profile_batches = []
    _it = iter(s3_loader)
    for _ in range(_need_batches):
        profile_batches.append(next(_it))
    print(
        f"S3: {dataset_name}  cached_batches={len(profile_batches)}  "
        f"batch_size={BATCH_SIZE}  num_workers={PROFILE_NUM_WORKERS}  num_classes={NUM_CLASSES}"
    )
else:
    raise ValueError(f"Unknown PROFILE_DATA_SOURCE={PROFILE_DATA_SOURCE!r}; use 'synthetic' or 's3'")

Synthetic: 10 batches  batch_size=2  num_classes=16


## 4. Model Initialization


In [7]:
meta_model = MetaModel(
    run_name=cfg.run_name,
    num_classes=NUM_CLASSES,
    device=str(device),
    model_kwargs=copy.deepcopy(cfg_model),
    training_mode=cfg.training_mode,
    training_kwargs=copy.deepcopy(cfg_training),
)
meta_model.model.freeze_encoder()
meta_model.model.train(True)
print("Model initialized, encoder frozen.")

Model initialized, encoder frozen.


In [8]:
total_params = sum(p.numel() for p in meta_model.model.parameters())
trainable_params = sum(p.numel() for p in meta_model.model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

In [9]:
print(f"Total parameters:     {total_params:>14,}")
print(f"Trainable parameters: {trainable_params:>14,}")
print(f"Frozen parameters:    {frozen_params:>14,}")

Total parameters:         85,672,720
Trainable parameters:         12,304
Frozen parameters:        85,660,416


## 5. Warm-up

Prime CUDA kernels, JIT caches, and cuDNN autotuner before profiling.


In [10]:
meta_model.model.train(True)

for batch in profile_batches[:N_WARMUP]:
    batch = prepare_batch(batch)
    loss, _, _, _ = meta_model.batch_predict_loss(batch)
    loss.backward()
    meta_model.optimizer.step()
    meta_model.optimizer.zero_grad()

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats(device)

print(f"Warm-up complete: {N_WARMUP} steps")

Warm-up complete: 5 steps


## 6. torch.profiler: Training Step Profile

Schedule: `wait=1` (skip), `warmup=1` (collect but discard), `active=3` (trace captured).
Each training phase is annotated with `record_function` so it appears as a named
region in the Chrome and TensorBoard traces.

In [11]:
activities = [torch.profiler.ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(torch.profiler.ProfilerActivity.CUDA)

meta_model.model.train(True)

with torch.profiler.profile(
    schedule=torch.profiler.schedule(wait=WAIT, warmup=WARMUP, active=ACTIVE, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler(str(PROFILER_LOG_DIR)),
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
    activities=activities,
) as prof:
    for batch in profile_batches[: WAIT + WARMUP + ACTIVE]:
        with torch.profiler.record_function("data_loading"):
            batch = prepare_batch(batch)

        with torch.profiler.record_function("forward"):
            loss, _, _, _ = meta_model.batch_predict_loss(batch)

        with torch.profiler.record_function("backward"):
            loss.backward()

        with torch.profiler.record_function("optimizer_step"):
            meta_model.optimizer.step()
            meta_model.optimizer.zero_grad()
            if torch.cuda.is_available():
                torch.cuda.synchronize()

        prof.step()

print("Profiling complete.")

STAGE:2026-04-03 13:50:22 6138:84932 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
[W CPUAllocator.cpp:235] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event
STAGE:2026-04-03 13:50:28 6138:84932 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-04-03 13:50:28 6138:84932 ActivityProfilerController.cpp:324] Completed Stage: Post Processing


Profiling complete.


## 6b. Reading `profiler_logs`

**TensorBoard + PyTorch Profiler plugin:** Kineto traces land in subfolders under `PROFILER_LOG_DIR` from
`tensorboard_trace_handler`. Run `tensorboard --logdir=<PROFILER_LOG_DIR>` (`torch-tb-profiler` is in the
`notebooks` dependency group). On SageMaker Studio, use Jupyter TensorBoard integration or port-forward.

**Chrome / Perfetto:** Timestamped JSON traces are written in **Section 11. Export**; open in Chrome `chrome://tracing` or [Perfetto](https://ui.perfetto.dev/).

**Printed tables (`key_averages`):** Sort by CUDA time for GPU kernels (ViT attention, `addmm`, `upsample_bilinear2d`).
Shape-grouped view links cost to tensor geometry (token grid vs full-res). DINOv3 uses five prefix tokens; patch
features are `[:, 5:]` before the segmentation head and upsample to `input_size`.

The **artifact table** cells (immediately after Section 11) list paths under `PROFILER_LOG_DIR`. Run them once after
the export cell so the Chrome/Perfetto JSON appears alongside TensorBoard folders.


## 7. Analysis: Operator Table (CUDA Time)

Top operators by total CUDA execution time.
Expect `aten::mm` / attention matmuls and the bilinear interpolate upsample to dominate.

In [12]:
sort_by_cuda = "cuda_time_total" if torch.cuda.is_available() else "cpu_time_total"
cuda_table = prof.key_averages().table(sort_by=sort_by_cuda, row_limit=20)

In [13]:
print(cuda_table)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.01%     306.000us       100.00%        5.250s        1.750s           4 b           0 b             3  
                                                forward        11.47%     602.070ms        97.56%        5.122s        1.707s     114.09 Mb      -6.69 Gb             3  
                                           aten::linear         0.12%       6.086ms        47.45%        2.491s      11.532ms       1.91 Gb           

## 8. Analysis: Operator Table (CPU Time)

Top operators by CPU time — highlights Python overhead,
host-side dispatch cost, and operators that are CPU-bound.

In [14]:
cpu_table = prof.key_averages().table(sort_by="cpu_time_total", row_limit=20)

In [15]:
print(cpu_table)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.01%     306.000us       100.00%        5.250s        1.750s           4 b           0 b             3  
                                                forward        11.47%     602.070ms        97.56%        5.122s        1.707s     114.09 Mb      -6.69 Gb             3  
                                           aten::linear         0.12%       6.086ms        47.45%        2.491s      11.532ms       1.91 Gb           

## 9. Analysis: Memory Profiling

Top allocators by self CUDA memory usage. Use this to identify which operators
hold the most memory at peak — useful for deciding whether batch size can be increased.

In [16]:
mem_sort_key = "self_cuda_memory_usage" if torch.cuda.is_available() else "self_cpu_memory_usage"
memory_table = prof.key_averages().table(sort_by=mem_sort_key, row_limit=15)

if torch.cuda.is_available():
    peak_memory_mb = torch.cuda.max_memory_allocated(device) / 1e6
    total_memory_mb = torch.cuda.get_device_properties(device).total_memory / 1e6

In [17]:
print(memory_table)
if torch.cuda.is_available():
    print(f"\nPeak GPU memory allocated: {peak_memory_mb:.1f} MB")
    print(f"Total GPU memory:          {total_memory_mb:.1f} MB")
    print(f"Memory utilization:        {100 * peak_memory_mb / total_memory_mb:.1f}%")

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            aten::addmm        39.59%        2.079s        42.41%        2.227s      12.370ms       1.70 Gb       1.70 Gb           180  
                                              aten::mul         2.81%     147.677ms         2.82%     148.093ms     658.191us       1.27 Gb       1.27 Gb           225  
                                            aten::empty         0.03%       1.397ms         0.03%       1.397ms       2.364us     964.41 Mb     964.41

## 10. Analysis: Shape-Grouped Table

Groups the same operator by input tensor shape. Reveals the cost difference between the
compact token grid (32×32) convolutions and the full-resolution (512×512) operations
such as the bilinear upsample and the loss computation.

In [18]:
shape_table = prof.key_averages(group_by_input_shape=True).table(sort_by=sort_by_cuda, row_limit=15)

In [19]:
print(shape_table)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls                                                                      Input Shapes  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  --------------------------------------------------------------------------------  
                                          ProfilerStep*         0.01%     306.000us       100.00%        5.250s        1.750s           4 b           0 b             3                                                                             

## 11. Export Traces

TensorBoard/Kineto output is already under `PROFILER_LOG_DIR` from `tensorboard_trace_handler`.
Each run writes a **timestamped** Chrome/Perfetto JSON so re-running this cell avoids
"Trace is already saved".

In [20]:
chrome_trace_path = PROFILER_LOG_DIR / f"chrome_trace_{datetime.now():%Y%m%d_%H%M%S}.json"
prof.export_chrome_trace(str(chrome_trace_path))

print(f"Chrome / Perfetto: {chrome_trace_path.resolve()}")
print("  Chrome:   chrome://tracing  -> Load")
print("  Perfetto: https://ui.perfetto.dev/")
print()
print(f"TensorBoard logdir: {PROFILER_LOG_DIR.resolve()}")
print(f"  tensorboard --logdir={PROFILER_LOG_DIR.resolve()}")
print("  Requires: torch-tb-profiler (see dependency group `notebooks`)")

RuntimeError: Trace is already saved.

In [21]:
import pandas as pd

_rows = []
for p in sorted(PROFILER_LOG_DIR.rglob("*")):
    rel = p.relative_to(PROFILER_LOG_DIR)
    _rows.append({"artifact": str(rel), "kind": "dir" if p.is_dir() else "file"})
if not _rows:
    _rows = [{"artifact": "(empty)", "kind": "-"}]
_profiler_artifacts_df = pd.DataFrame(_rows)

In [22]:
from great_tables import GT

GT(_profiler_artifacts_df)

artifact,kind
4e:f0:70:01:bd:45_2494.1775234102072877000.pt.trace.json,file
4e:f0:70:01:bd:45_6138.1775238628765427000.pt.trace.json,file


## 12. Microbenchmarks

Use `torch.utils.benchmark.Timer` to isolate and measure individual sub-components
of the forward pass. Each timer runs for at least 2 seconds to produce statistically
stable median + IQR estimates with automatic CUDA synchronization.

In [ ]:
x_bench = torch.randn(BATCH_SIZE, 3, H, W, device=device)
batch_bench = (
    x_bench,
    torch.randint(0, NUM_CLASSES, (BATCH_SIZE, H, W), dtype=torch.long, device=device),
)

meta_model.model.eval()
with torch.no_grad():
    enc_out = meta_model.model.encoder(x_bench)
    patch_embeddings = enc_out.last_hidden_state[:, 5:].clone()

token_h = meta_model.model.token_height
token_w = meta_model.model.token_width
logits_bench = torch.randn(BATCH_SIZE, NUM_CLASSES, token_h, token_w, device=device)

meta_model.model.train(True)
print(f"Patch embeddings: {patch_embeddings.shape}")
print(f"Token grid:       {token_h}x{token_w}")

In [ ]:
@torch.no_grad()
def _bench_encoder():
    return meta_model.model.encoder(x_bench)


@torch.no_grad()
def _bench_head():
    return meta_model.model.head(patch_embeddings)


@torch.no_grad()
def _bench_upsample():
    return F.interpolate(logits_bench, size=(H, W), mode="bilinear", align_corners=False)


def _bench_full_forward():
    loss, _, _, _ = meta_model.batch_predict_loss(batch_bench)
    meta_model.optimizer.zero_grad()
    return loss


# Compare groups by `label` and uses `description` as column headers; all-None descriptions
# break Table.render() (None in the header row). Use one label + distinct descriptions.
_bench_sub = f"batch {BATCH_SIZE}, {H}x{W}"
bench_results = [
    benchmark.Timer(
        stmt="f()",
        globals={"f": _bench_encoder},
        label="Microbenchmark",
        sub_label=_bench_sub,
        description="DINOv3 encoder",
    ).blocked_autorange(min_run_time=2.0),
    benchmark.Timer(
        stmt="f()",
        globals={"f": _bench_head},
        label="Microbenchmark",
        sub_label=_bench_sub,
        description="Linear head",
    ).blocked_autorange(min_run_time=2.0),
    benchmark.Timer(
        stmt="f()",
        globals={"f": _bench_upsample},
        label="Microbenchmark",
        sub_label=_bench_sub,
        description="Bilinear upsample",
    ).blocked_autorange(min_run_time=2.0),
    benchmark.Timer(
        stmt="f()",
        globals={"f": _bench_full_forward},
        label="Microbenchmark",
        sub_label=_bench_sub,
        description="Full forward + loss",
    ).blocked_autorange(min_run_time=2.0),
]

In [ ]:
compare = benchmark.Compare(bench_results)
compare.print()